# BioLORD (only includes Leiden)

In [2]:
from sentence_transformers import SentenceTransformer

with open("../data/mouse_files/gene_set_list_mouse_2024_clean.txt", "r") as f:
    phenotypes = f.readlines()

phenotypes = [p.strip() for p in phenotypes]

model = SentenceTransformer('FremyCompany/BioLORD-2023')
embeddings = model.encode(phenotypes, normalize_embeddings=True)
similarities = model.similarity(embeddings, embeddings)
similarities_np = similarities.numpy()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1192.36it/s, Materializing param=pooler.dense.weight]                       
MPNetModel LOAD REPORT from: FremyCompany/BioLORD-2023
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import numpy as np

#save txt file of embeddings
np.savetxt("biolord_embedding.txt", embeddings)

In [8]:
X = np.asarray(embeddings)
print(X.shape)

(12624, 768)


In [9]:
import numpy as np
from sklearn.neighbors import NearestNeighbors
from scipy import sparse
from sknetwork.clustering import Leiden

X = np.asarray(embeddings)

k = 75

# Build kNN graph
nn = NearestNeighbors(n_neighbors=k + 1, metric="cosine")
nn.fit(X)
distances, indices = nn.kneighbors(X)

# Remove self-neighbor
distances = distances[:, 1:]
indices = indices[:, 1:]

n = X.shape[0]
rows = np.repeat(np.arange(n), k)
cols = indices.flatten()
sims = (1 - distances).flatten()

A = sparse.csr_matrix((sims, (rows, cols)), shape=(n, n))

# Make graph undirected
A = A.maximum(A.T)
A.eliminate_zeros()

leiden = Leiden(sort_clusters=True)
labels_leiden = leiden.fit_predict(A)

cluster_probs = leiden.fit_predict_proba(A)
print(labels_leiden.shape)         # (n_samples,)
print(cluster_probs.shape)  # (n_samples, n_clusters)

(12624,)
(12624, 15)


In [11]:
epsilon = 1e-12  # avoid log(0)

entropy = -np.sum(cluster_probs * np.log(cluster_probs + epsilon), axis=1)
avg_entropy = np.mean(entropy)
print(avg_entropy)

0.6818515422771082


In [ ]:
np.save("../data/mouse_files/entropy_biolord.npy", entropy)

## Files to save

In [ ]:
with open("../data/mouse_files/biolord_leiden_clusters.txt", "w") as f:
    for name, label in zip(phenotypes, labels_leiden):
        clean_name = name.replace("\n", "")
        f.write(f"{clean_name}\t{label}\n")

In [ ]:
import umap
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook" #to render correctly in notebook

# UMAP projection
umap_model = umap.UMAP(
    n_components=2,
    min_dist=0.2
)

X_umap = umap_model.fit_transform(X)

# Plot
fig = px.scatter(
    x=X_umap[:,0],
    y=X_umap[:,1],
    color=labels_leiden.astype(str),
    hover_name=phenotypes
)

fig.update_layout(
    scene=dict(
        xaxis_title="UMAP 1",
        yaxis_title="UMAP 2",
    ),
    title="UMAP of Gene Set Embeddings (Leiden clusters)"
)

fig.show()